In [7]:
import pvlib
import pandas as pd
import numpy as np
import os
from glob import glob

# Percorso della cartella contenente i file CSV
folder_path = r'C:\Users\vitto\Desktop\DATI SBAGLIATI'  # Modifica con il tuo percorso

# Trova tutti i file CSV nella cartella
file_paths = glob(os.path.join(folder_path, '*.csv'))

# Funzione per calcolare la radiazione extraterrestre per un file specifico
def aggiungi_radiazione_extraterrestre(file_path, latitude, longitude, altitude):
    # Crea l'oggetto Location
    location = pvlib.location.Location(latitude, longitude, altitude=altitude)

    # Legge il file CSV
    df = pd.read_csv(file_path, delimiter=';', decimal='.')

    # Converte la colonna "data" e "ora" in un datetime unico
    df['data_ora'] = pd.to_datetime(df['data'] + ' ' + df['ora'], format='mixed')
    df = df.set_index('data_ora').tz_localize('Europe/Rome', nonexistent='shift_forward',ambiguous='NaT').tz_convert('UTC')
    
    # Rimuovi gli orari ambigui con valori 'NaT'
    df = df.dropna()

    # Calcola la posizione solare per ogni ora
    solar_position = location.get_solarposition(df.index)

    # Calcola la radiazione extraterrestre
    extra_radiation = pvlib.irradiance.get_extra_radiation(df.index)

    # Calcola l'angolo zenitale del sole in radianti
    zenith_angle_radians = np.radians(solar_position['apparent_zenith'])

    # Calcola la radiazione extraterrestre orizzontale (GHI extra)
    ghi_extra = extra_radiation * np.cos(zenith_angle_radians)

    # Imposta a zero i valori negativi (quando il sole è sotto l'orizzonte)
    ghi_extra[ghi_extra < 0] = 0

    # Aggiunge la nuova colonna al DataFrame originale
    df['rad extrat'] = ghi_extra.values

    # Salva i risultati nel file CSV originale
    df.to_csv(file_path, index=False, sep=',', decimal='.')
    print(f"Radiazione extraterrestre aggiunta al file: {file_path}")

# Inserisci manualmente le coordinate (latitudine, longitudine, altitudine) per ogni file
coordinate_files = {
    #'ACQUI TERME NI.csv': {'latitude': 44.68, 'longitude': 8.48, 'altitude': 250},
    #'ALBA.csv': {'latitude': 44.41, 'longitude': 8.01, 'altitude': 300},
    #'ALESSANDRIA.csv': {'latitude': 44.91, 'longitude': 8.61, 'altitude': 250},
    #'ARQUATA_SCRIVIA.csv': {'latitude': 44.68, 'longitude': 8.88, 'altitude': 300},
    #'ASTI.csv': {'latitude': 44.90, 'longitude': 8.21, 'altitude': 250},
    #'AVIGLIANA.csv': {'latitude': 45.09, 'longitude': 7.39, 'altitude': 300},
    #'BASALUZZO.csv': {'latitude': 44.76, 'longitude': 8.68, 'altitude': 250},
    #'BORGOFRANCO_DIVREA.csv': {'latitude': 45.51, 'longitude': 7.85, 'altitude': 300},
    #'BORGOMANERO': {'latitude': 45.68, 'longitude': 8.46, 'altitude': 250},
    #'BRA.csv': {'latitude': 44.70, 'longitude': 7.85, 'altitude': 300},
    #'BUTTIGLIERA_DASTI SI.csv': {'latitude': 45.02, 'longitude': 7.93, 'altitude': 250},
    #'CANDIA_CANAVESE.csv': {'latitude': 45.33, 'longitude': 7.89, 'altitude': 300},
    #'CARMAGNOLA.csv': {'latitude': 44.89, 'longitude': 7.69, 'altitude': 250},
    #'CASALE_MONFERRATO.csv': {'latitude': 45.13, 'longitude': 8.50, 'altitude': 300},
    #'CASTELLAR_PONZANO .csv': {'latitude': 44.82, 'longitude': 8.85, 'altitude': 250},
    #'COSTIGLIOLE_SALUZZO.csv': {'latitude': 44.55, 'longitude': 7.50, 'altitude': 300},
    #'CUMIANA.csv': {'latitude': 44.96, 'longitude': 7.39, 'altitude': 250},
    #'CUNEO.csv': {'latitude': 44.38, 'longitude': 7.54, 'altitude': 300},
    #'FOSSANO.csv': {'latitude': 44.54, 'longitude': 7.79, 'altitude': 250},#SONO ARRIVATO QUI CON LE COORDINATE
    #'ISOLA_SANTANTONIO.csv': {'latitude': 45.01, 'longitude': 8.01, 'altitude': 300},
    #'LOZZOLO.csv': {'latitude': 45.64, 'longitude': 8.33, 'altitude': 250},
    #'MOMBALDONE.csv': {'latitude': 44.57, 'longitude': 8.33, 'altitude': 300},
    #'MOMBARCARO.csv': {'latitude': 44.47, 'longitude': 8.09, 'altitude': 250},
    #'MONCALIERI.csv': {'latitude': 45.00, 'longitude': 7.68, 'altitude': 300},
    #'MONTECHIARO_DASTI SI.csv': {'latitude': 45.01, 'longitude': 8.10, 'altitude': 250},
    #'NOVI_LIGURE.csv': {'latitude': 44.48, 'longitude': 8.76, 'altitude': 300},
    #'PINO_TORINESE.csv': {'latitude': 45.04, 'longitude': 7.77, 'altitude': 250},
    #'PONZONE.csv': {'latitude': 44.59, 'longitude': 8.46, 'altitude': 300},
    #'SALICETO.csv': {'latitude': 44.41, 'longitude': 8.16, 'altitude': 250},
    #'SARDIGLIANO.csv': {'latitude': 44.76, 'longitude': 8.91, 'altitude': 300},
    # 'TORINO.csv': {'latitude': 45.07, 'longitude': 7.69, 'altitude': 250},
    #'TREISO.csv': {'latitude': 44.68, 'longitude': 8.08, 'altitude': 300},
     #'VERCELLI.csv': {'latitude': 45.32, 'longitude': 8.39, 'altitude': 250},
    #'VEROLOENGO.csv': {'latitude': 45.19, 'longitude': 8.01, 'altitude': 300},
     #'VILLANOVA_SOLARO.csv': {'latitude': 44.73, 'longitude': 7.57, 'altitude': 250},

    'BORGOMANERO.csv': {'latitude': 45.68, 'longitude': 8.46, 'altitude': 250},
    'CASTELLAR_PONZANO.csv': {'latitude': 44.82, 'longitude': 8.85, 'altitude': 250},
    'MONTECHIARO_DASTI.csv': {'latitude': 45.01, 'longitude': 8.10, 'altitude': 250},
    'VEROLOENGO.csv': {'latitude': 45.19, 'longitude': 8.01, 'altitude': 300}

}# Itera su ogni file nella cartella
for file_path in file_paths:
    file_name = os.path.basename(file_path)
    if file_name in coordinate_files:
        coords = coordinate_files[file_name]
        aggiungi_radiazione_extraterrestre(file_path, coords['latitude'], coords['longitude'], coords['altitude'])
    else:
        print(f"Coordinate non trovate per il file: {file_name}. Saltato.")

print("Processo completato.")


KeyError: 'data'